In [10]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join('../..')))

In [11]:
from src.patient_data_dispatcher import PatientDataDispatcher
from src.core.enums import MileStone, DataFrequency, PatientDataType

import torch

In [12]:
dmo_features = [
    "wb_all_sum",
    "walkdur_all_sum",
    "wbsteps_all_sum",
    "wbdur_all_avg",
    "wbdur_all_p90",
    "wbdur_all_var",
    "cadence_all_avg",
    "strdur_all_avg",
    "cadence_all_var",
    "strdur_all_var",
    "ws_1030_avg",
    "strlen_1030_avg",
    "wb_10_sum",
    "ws_10_p90",
    "wb_30_sum",
    "ws_30_avg",
    "strlen_30_avg",
    "cadence_30_avg",
    "strdur_30_avg",
    "ws_30_p90",
    "cadence_30_p90",
    "ws_30_var",
    "strlen_30_var",
    "wb_60_sum",
    "total_worn_h",
]

In [13]:
static_features = [
    "weight",
    "height",
    "gender",
    "EDFSCR1L"
]

In [14]:
pdd = PatientDataDispatcher(
    "../../config/config.yaml",
    dmo_features,
    MileStone.ALL,
    data_frequency=DataFrequency.DAILY,
    filtered=True,
    #static_features=static_features,
)
ids = list(set(pdd.metadata["Local.Participant"].to_list()))
dmo_data, dmo_labels = pdd.get_patient_data(PatientDataType.MILESTONE, ids=ids)

In [15]:
print(dmo_data.shape)

torch.Size([593, 5, 7, 25])


In [ ]:
patients, visits, days, features = dmo_data.shape
dmo_data_reshaped = dmo_data.reshape(patients * visits, days * features)

mask = (dmo_data_reshaped == -1)
rows_with_missing = torch.any(mask, axis = 1)
total_incomplete_visits = torch.sum(rows_with_missing)
print(f"Number of visits with missing data: {total_incomplete_visits}")
print(f"Number of visits in total: {patients * visits}")
print(f"complete visits {(patients * visits) - total_incomplete_visits}")

dmo_data_reshaped = dmo_data.reshape(patients, visits * days * features)

mask = (dmo_data_reshaped == -1)
rows_with_missing = torch.any(mask, axis = 1)
total_incomplete_patients = torch.sum(rows_with_missing)
print(f"Number of patients with missing data: {total_incomplete_patients}")
print(f"Number of patients with no missing data: {patients - total_incomplete_patients}")

Number of visits with missing data: 1917
Number of visits in total: 2965
complete visits 1048
Number of patients with missing data: 554
Number of patients with no missin data: 39


In [17]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# perform impute on a visit by visit basis, as some visits are completely missing

imputer = IterativeImputer(missing_values=-1, tol=1e-2, keep_empty_features=True)

for p in range(patients):
    for v in range(visits):
        visit_data = dmo_data[p, v]

        if (visit_data == -1).all():
            continue

        dmo_data[p, v] = torch.from_numpy(imputer.fit_transform(visit_data)).to(dtype=torch.float64)

/home/gwilym-rutherford/Documents/Year 3 Tuos/dissertation/Workspace/Experiment 1/.venv/lib/python3.13/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [ ]:
patients, visits, days, features = dmo_data.shape
dmo_data_reshaped = dmo_data.reshape(patients * visits, days * features)

mask = (dmo_data_reshaped == -1)
rows_with_missing = torch.any(mask, axis = 1)
total_incomplete_visits = torch.sum(rows_with_missing)
print(f"Number of visits with missing data: {total_incomplete_visits}")
print(f"Number of visits in total: {patients * visits}")
print(f"complete visits {(patients * visits) - total_incomplete_visits}")

dmo_data_reshaped = dmo_data.reshape(patients, visits * days * features)

mask = (dmo_data_reshaped == -1)
rows_with_missing = torch.any(mask, axis = 1)
total_incomplete_patients = torch.sum(rows_with_missing)
print(f"Number of patients with missing data: {total_incomplete_patients}")
print(f"Number of patients with no missing data: {patients - total_incomplete_patients}")

Number of visits with missing data: 676
Number of visits in total: 2965
complete visits 2289
Number of patients with missing data: 336
Number of patients with no missin data: 257
